# Install checker log comparison

Compares two `Test-AppInstalled` check logs, including per-app entries for not-installed members, and looks up changed app IDs in the configured app JSON to report categories and per-section category counts.

In [ ]:
import ujson as json
import re
from pathlib import Path

LOG_DIR = Path.cwd().parent / "logs" / "check"
FIRST_LOG = LOG_DIR / "2026-10-20_11-26-41.log"
SECOND_LOG = LOG_DIR / "2026-10-20_22-23-03.log"
APP_JSON = LOG_DIR.parent.parent / "results" / "app_2026-10-20.json"
with APP_JSON.open(encoding="utf-8") as f:
    apps_by_id = {app["itemID"].lower(): app for app in json.load(f)}

ITEM_RE = re.compile(r"^\s*-\s+(?P<name>.*?)\s+\[(?P<id>[^\]]+)\]\s*$")
APP_CHECK_RE = re.compile(r"^\s*\[\d+/\d+\]\s+(?P<label>.+?)\s*$")
APP_LABEL_RE = re.compile(r"^(?P<name>.*?)\s+\[(?P<id>[^\]]+)\]$")
COUNT_RE = re.compile(r"^(?P<key>[^:]+?)\s*:\s*(?P<value>\d+)\s*$")


def read_log(path):
    raw = path.read_bytes()
    if raw.startswith((b"\xff\xfe", b"\xfe\xff")):
        return raw.decode("utf-16")
    return raw.decode("utf-8-sig", errors="replace")


def parse_summary(path):
    lines = read_log(path).splitlines()
    start = next(i for i, line in enumerate(lines) if line.strip() == "SUMMARY")

    counts = {}
    lists = {"installed": {}, "stale": {}, "not_installed": {}}
    current_app = None
    for line in lines[:start]:
        app_check = APP_CHECK_RE.match(line)
        if app_check:
            label = app_check["label"]
            app_label = APP_LABEL_RE.match(label)
            if app_label:
                current_app = (app_label["id"].lower(), app_label["name"])
            else:
                current_app = (label.lower(), label)
        elif current_app and (
            "No installed package found matching input criteria" in line
            or "No packages found matching input criteria" in line
        ):
            app_id, name = current_app
            lists["not_installed"][app_id] = name

    current = None

    for line in lines[start + 1:]:
        stripped = line.strip()
        if stripped.startswith("Installed apps"):
            current = "installed"
            continue
        if stripped.startswith("Stale apps"):
            current = "stale"
            continue
        if stripped.startswith("Not installed apps"):
            current = "not_installed"
            continue

        item = ITEM_RE.match(line)
        if item and current:
            lists[current][item["id"].lower()] = item["name"]
            continue

        count = COUNT_RE.match(stripped)
        if count:
            counts[count["key"]] = int(count["value"])

    return {
        "total": counts.get("Total apps checked"),
        "installed_count": counts.get("Installed"),
        "not_installed_count": counts.get("Not installed"),
        "stale_count": counts.get("Stale (winget only)"),
        "installed": lists["installed"],
        "stale": lists["stale"],
        "not_installed": lists["not_installed"],
    }


def print_diff(title, left, right, include_categories=True):
    only = {app_id: name for app_id, name in left.items() if app_id not in right}
    print(f"{title} ({len(only)})")
    if not only:
        print("  (none)")
        return

    category_counts = {}
    for app_id, name in only.items():
        print(f"  - {name} [{app_id}]")
        if not include_categories:
            continue
        app = apps_by_id.get(app_id.lower())
        if app is None:
            print(f"      Categories: ID not found in {APP_JSON.name}")
            continue

        categories = app["categories"]
        if categories:
            for category in categories:
                category_counts[category] = category_counts.get(category, 0) + 1
            # print(f"      Categories: {', '.join(categories)}")
        else:
            print("      Categories: (none listed)")

    if not include_categories:
        return

    print("  Category totals (apps per category):")
    if category_counts:
        for category, count in sorted(category_counts.items()):
            print(f"    {category}: {count}")
    else:
        print("    (no category matches)")


first = parse_summary(FIRST_LOG)
second = parse_summary(SECOND_LOG)

print(f"First : {FIRST_LOG.name} (apps checked: {first['total']})")
print(f"Second: {SECOND_LOG.name} (apps checked: {second['total']})")

## 1. Total installed

In [ ]:
print(f"First : {first['installed_count']}")
print(f"Second: {second['installed_count']}")
print(f"Diff  : {second['installed_count'] - first['installed_count']:+d}")

## 2. Installed in first file but not in second

In [ ]:
print_diff("Installed only in first", first["installed"], second["installed"])

## 3. Installed in second file but not in first

In [ ]:
print_diff("Installed only in second", second["installed"], first["installed"])

## 4. Total not installed

In [ ]:
print(f"First : {first['not_installed_count']}")
print(f"Second: {second['not_installed_count']}")
print(f"Diff  : {second['not_installed_count'] - first['not_installed_count']:+d}")

## 5. Not installed in the first file but not in the second

In [ ]:
print_diff("Not installed only in first", first["not_installed"], second["not_installed"], include_categories=False)

## 6. Not installed in the second file but not in the first

In [ ]:
print_diff("Not installed only in second", second["not_installed"], first["not_installed"], include_categories=False)

## 7. Total stale

In [ ]:
print(f"First : {first['stale_count']}")
print(f"Second: {second['stale_count']}")
print(f"Diff  : {second['stale_count'] - first['stale_count']:+d}")

## 8. Stale in first file but not in second

In [ ]:
print_diff("Stale only in first", first["stale"], second["stale"])

## 9. Stale in second file but not in first

In [ ]:
print_diff("Stale only in second", second["stale"], first["stale"])